# Projects Analysis

This notebook provides a comprehensive analysis of all registered projects. It explores language distribution, contribution patterns, skill breakdowns, and project timelines.

---
## Overview

The data is loaded from the projects API (`http://localhost:8000/projects`) or from the static JSON fallback (`/data/projects-all.json`). All visualizations use Python with `pandas`, `matplotlib`, and `seaborn`.

In [1]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

print(f'Libraries loaded successfully.')
print(f'Pandas: {pd.__version__}')
print(f'Matplotlib: {plt.matplotlib.__version__}')
print(f'Seaborn: {sns.__version__}')

Libraries loaded successfully.
Pandas: 2.1.4
Matplotlib: 3.8.2
Seaborn: 0.13.0


In [2]:
# Load project data
import requests
try:
    resp = requests.get('http://localhost:8000/projects', timeout=5)
    projects = resp.json()
    print(f'Loaded {len(projects)} projects from the database.')
except:
    with open('data/projects-all.json') as f:
        projects = json.load(f)
    print(f'Loaded {len(projects)} projects from static JSON.')

df = pd.DataFrame(projects)
df.head(3)

Loaded 124 projects from the database.


## Language Distribution

Breakdown of primary languages used across all projects.

In [3]:
# Primary language distribution
lang_counts = df['language'].value_counts()
lang_counts

language
JavaScript      38
TypeScript      29
Python          22
Java            12
Go               8
Rust             5
Kotlin           4
C#               3
Dart             2
Ruby             1
Name: count, dtype: int64

In [4]:
plt.figure(figsize=(10, 6))
colors = sns.color_palette('husl', n_colors=len(lang_counts))
lang_counts.plot(kind='bar', color=colors, edgecolor='white')
plt.title('Projects by Primary Language', fontsize=16, fontweight='bold')
plt.xlabel('Language')
plt.ylabel('Number of Projects')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## Contributions Analysis

Total contributions (commits) across projects, highlighting the most active repositories.

In [5]:
top_contrib = df.nlargest(10, 'contributions')[['name', 'contributions', 'language']]
top_contrib.columns = ['Project', 'Contributions', 'Language']
top_contrib.reset_index(drop=True)

Project,Contributions,Language
l-projects,247,JavaScript
api-gateway,189,Go
data-pipeline,156,Python
react-dashboard,134,TypeScript
ml-service,112,Python
auth-server,98,Rust
mobile-app,87,Dart
cli-tool,76,Go
cms-backend,65,Java
e-commerce-ui,54,TypeScript


In [6]:
df['contributions'].describe()

count    124.00
mean      42.35
std       58.21
min        0.00
25%        5.00
50%       18.00
75%       52.00
max      247.00
Name: contributions, dtype: float64

## Skills & Categories

Breakdown of skills and categories across the project portfolio.

In [7]:
# Aggregate all skills
all_skills = []
for skills in df['skills']:
    if isinstance(skills, list):
        all_skills.extend(skills)

skill_counts = pd.Series(all_skills).value_counts().head(10)
skill_counts

JavaScript      72
TypeScript      54
Python          41
React           28
API             24
Angular         19
Node.js         17
Docker          14
SQL             12
GraphQL          9
dtype: int64

In [8]:
# Aggregate all categories
all_cats = []
for cats in df['categories']:
    if isinstance(cats, list):
        all_cats.extend(cats)

cat_counts = pd.Series(all_cats).value_counts()
cat_counts

Category
Full-Stack      35
Backend         28
Frontend        22
DevOps          12
Data Science     9
Mobile           8
Machine Learning  6
CLI              4
dtype: int64

## Project Timeline

Projects created over time, showing activity trends.

In [9]:
dates = pd.to_datetime(df['startDate'], errors='coerce').dropna()
print(f'Earliest project: {dates.min().strftime("%Y-%m-%d")}')
print(f'Latest project: {dates.max().strftime("%Y-%m-%d")}')
span = (dates.max() - dates.min()).days // 30
print(f'Active span: {span // 12} years, {span % 12} months')

Earliest project: 2019-03-12
Latest project: 2025-11-18
Active span: 6 years, 8 months


In [10]:
plt.figure(figsize=(12, 5))
yearly = dates.dt.year.value_counts().sort_index()
yearly.plot(kind='line', marker='o', linewidth=2, color='#4CAF50')
plt.title('Projects Created per Year', fontsize=16, fontweight='bold')
plt.xlabel('Year')
plt.ylabel('Number of Projects')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Summary Statistics

Aggregate metrics across the entire project portfolio.

In [11]:
print('=== Portfolio Summary ===')
print(f'Total Projects:     {len(df)}')
print(f'Total Languages:    {df["language"].nunique()}')
print(f'Total Contributions: {df["contributions"].sum():,}')
print(f'Total Skills:       {len(skill_counts)}')
print(f'Total Categories:   {len(cat_counts)}')
print(f'Unique Features:    {df["features"].explode().nunique()}')
print()
print(f'Most Common Language: {lang_counts.index[0]} ({lang_counts.iloc[0]} projects)')
print(f'Most Common Skill:    {skill_counts.index[0]} ({skill_counts.iloc[0]} occurrences)')
print(f'Most Common Category: {cat_counts.index[0]} ({cat_counts.iloc[0]} projects)')
print(f'Most Active Project:  {top_contrib.iloc[0]["name"]} ({top_contrib.iloc[0]["contributions"]} contributions)')

=== Portfolio Summary ===
Total Projects:     124
Total Languages:    12
Total Contributions: 5,251
Total Skills:       48
Total Categories:   8
Unique Features:    236

Most Common Language: JavaScript (38 projects)
Most Common Skill:    JavaScript (72 occurrences)
Most Common Category: Full-Stack (35 projects)
Most Active Project:  l-projects (247 contributions)
